<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">01_Demo_Required_Setup</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# Required Setup
### Duration - 2 Minutes

<p style="font-size: 1em; line-height: 1.6; color: #333">Run this notebook once before starting any demos or labs. It initializes your personal catalog, creates the <code>spark_dev</code> schema used throughout the course, and builds the raw JSON-string tables used in the complex data section by reshaping the Databricks <code>samples.bakehouse</code> dataset.</p>

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## Step 1 of 4 - Catalog Setup

Detects your workspace type and sets your personal catalog as the default. Creates a `labuser_<yourname>` catalog if it does not already exist.

In [0]:
%run ./Includes/Classroom-Setup-Common

## Step 2 of 4 - Create the Course Schema

Creates the `spark_dev` schema, the working schema for every table you create in this course.

In [0]:
-- Create the Course Schema
CREATE SCHEMA IF NOT EXISTS spark_dev;
USE SCHEMA spark_dev;

SELECT current_catalog() AS catalog, current_schema() AS schema;

## Step 3 of 4 - Build the Raw Customer Profiles Table

Reshapes the Bakehouse customers and transactions sample tables into `raw_customer_profiles`, a table where the profile, product, and purchase details are stored as **JSON strings** - the way data often arrives from an API or event feed. Converting these strings back into structured types is the subject of the *Working with Complex Data* demo.

In [0]:
%python
# Build the Raw Customer Profiles Table
from pyspark.sql.functions import col, concat_ws, struct, to_json, collect_list, collect_set

customers_df = spark.read.table("samples.bakehouse.sales_customers")
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

# Aggregate each customer's purchases into JSON array strings
purchases_df = (
    transactions_df
    .groupBy("customerID")
    .agg(
        to_json(collect_set("product")).alias("products"),
        to_json(
            collect_list(
                struct("product", "quantity", "totalPrice", "paymentMethod", "dateTime")
            )
        ).alias("purchases"),
    )
)

# Pack the customer profile attributes into a JSON object string
profiles_df = (
    customers_df
    .select(
        "customerID",
        concat_ws(" ", col("first_name"), col("last_name")).alias("name"),
        col("email_address").alias("email"),
        to_json(struct("city", "state", "country", "continent", "gender")).alias("profile"),
    )
)

raw_customer_profiles_df = profiles_df.join(purchases_df, "customerID", "inner")

raw_customer_profiles_df.write.mode("overwrite").saveAsTable("raw_customer_profiles")

print(f"Saved {spark.table('raw_customer_profiles').count()} records to the raw_customer_profiles table")

## Step 4 of 4 - Build the Raw Franchise Sales Table

Builds `raw_franchise_sales` from the Bakehouse franchises and transactions sample tables - the location details, payment methods, and recent sales are stored as JSON strings. This table is the source for the *Working with Complex Data* lab.

In [0]:
%python
# Build the Raw Franchise Sales Table
from pyspark.sql.functions import struct, to_json, collect_list, collect_set

franchises_df = spark.read.table("samples.bakehouse.sales_franchises")
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

# Aggregate each franchise's sales into JSON array strings
franchise_sales_df = (
    transactions_df
    .groupBy("franchiseID")
    .agg(
        to_json(collect_set("paymentMethod")).alias("payment_methods"),
        to_json(
            collect_list(struct("product", "quantity", "totalPrice", "dateTime"))
        ).alias("recent_sales"),
    )
)

# Pack the franchise location attributes into a JSON object string
raw_franchise_sales_df = (
    franchises_df
    .select(
        "franchiseID",
        "name",
        to_json(struct("city", "district", "country", "size")).alias("location"),
    )
    .join(franchise_sales_df, "franchiseID", "inner")
)

raw_franchise_sales_df.write.mode("overwrite").saveAsTable("raw_franchise_sales")

print(f"Saved {spark.table('raw_franchise_sales').count()} records to the raw_franchise_sales table")

In [0]:
-- Confirm Setup Complete
SELECT 'Required Setup complete. You may now run any demo or lab notebook.' AS status;

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>